In [13]:
import torch
import os, sys, json, gc, time, shutil, subprocess, glob
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, set_seed

In [14]:
set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Device: cpu


In [ ]:
GGUF_DIR    = "/llama-3.2-1B-Instruct-gguf"

In [16]:
try:
    from gguf import GGUFReader, LlamaFileType  # ships with the `gguf` pip package
    HAVE_GGUF_PY = True
except Exception as e:
    HAVE_GGUF_PY = False
    print("`gguf` Python package not installed — skipping metadata inspection.")
    print("  install with: pip install gguf")
    print(f"  reason: {e}")

In [17]:
Q4KM_PATH = os.path.join(GGUF_DIR, "Llama-3.2-1B-Instruct.Q4_K_M.gguf")
assert os.path.exists(Q4KM_PATH), f"Q4_K_M file not found: {Q4KM_PATH}"

if HAVE_GGUF_PY:
    reader = GGUFReader(Q4KM_PATH)
    n_tensors = len(reader.tensors)
    n_bytes   = sum(len(t.data) for t in reader.tensors)
    print(f"Q4_K_M GGUF tensors : {n_tensors}")
    print(f"Q4_K_M tensor bytes : {n_bytes/1024**2:.2f} MB (uncompressed view)")

    # First few tensor names — should look like llama layers
    print("\nFirst 10 tensors:")
    for t in reader.tensors[:10]:
        print(f"  {t.name:55s} shape={tuple(t.shape)} type={t.tensor_type.name}")
else:
    # Fall back: just print the file size and trust llama-quantize
    sz = os.path.getsize(Q4KM_PATH)
    print(f"Q4_K_M file exists: {Q4KM_PATH}")
    print(f"Q4_K_M file size : {sz/1024**3:.3f} GB ({sz/1024**2:.2f} MB)")

Q4_K_M GGUF tensors : 147
Q4_K_M tensor bytes : 0.55 MB (uncompressed view)

First 10 tensors:
  output_norm.weight                                      shape=(2048,) type=F32
  rope_freqs.weight                                       shape=(32,) type=F32
  token_embd.weight                                       shape=(2048, 128256) type=Q6_K
  blk.0.attn_k.weight                                     shape=(2048, 512) type=Q4_K
  blk.0.attn_norm.weight                                  shape=(2048,) type=F32
  blk.0.attn_output.weight                                shape=(2048, 2048) type=Q4_K
  blk.0.attn_q.weight                                     shape=(2048, 2048) type=Q4_K
  blk.0.attn_v.weight                                     shape=(2048, 512) type=Q6_K
  blk.0.ffn_down.weight                                   shape=(8192, 2048) type=Q6_K
  blk.0.ffn_gate.weight                                   shape=(2048, 8192) type=Q4_K


In [19]:
try:
    from llama_cpp import Llama
    HAVE_LLAMA_CPP = True
except Exception as e:
    HAVE_LLAMA_CPP = False
    print("`llama-cpp-python` not installed — skipping inference smoke test.")
    print("  install with: pip install llama-cpp-python")
    print(f"  reason: {e}")

if HAVE_LLAMA_CPP:
    n_threads = max(1, (os.cpu_count() or 4) - 1)
    n_ctx     = 2048
    n_gpu     = 1 if DEVICE == "cuda" else 0

    print(f"Loading Q4_K_M with n_ctx={n_ctx}, n_threads={n_threads}, n_gpu_layers={n_gpu}")
    t0 = time.time()
    llm = Llama(
        model_path=Q4KM_PATH,
        n_ctx=n_ctx,
        n_threads=n_threads,
        n_gpu_layers=n_gpu,
        verbose=False,
    )
    print(f"Loaded in {time.time()-t0:.1f} s")

    prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n" \
             "Rephrase this : Write assay in Federated learning.<|eot_id|>" \
             "<|start_header_id|>assistant<|end_header_id|>\n\n"
    t0 = time.time()
    out = llm(prompt, max_tokens=120, temperature=0.0, stop=["<|eot_id|>"])
    dt = time.time() - t0
    print(f"\nQ4_K_M reply ({dt:.2f} s):")
    print(out["choices"][0]["text"])

Loading Q4_K_M with n_ctx=2048, n_threads=31, n_gpu_layers=0
Loaded in 6.3 s


/home/virendra/.local/lib/python3.13/site-packages/llama_cpp/llama.py:1314: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(



Q4_K_M reply (4.28 s):
Here's a rephrased version:

"Write a model in Federated Learning"

Or, if you'd like a more specific rephrased version:

"Train a model using Federated Learning"

Or, if you'd like to emphasize the collaborative aspect:

"Collaborative model training using Federated Learning"

Or, if you'd like to highlight the decentralized nature of Federated Learning:

"Decentralized model training using Federated Learning"
